# Caso del actuador lineal

Este notebook utiliza datos sintéticos para practicar verificación de
afirmaciones técnicas. El modelo no representa un actuador industrial
específico y no valida un sistema real de diagnóstico.

**Objetivos**

- comprobar la procedencia y estructura de los datos;
- comparar señales medidas y señales de validación;
- decidir qué afirmaciones respaldan los resultados;
- documentar límites y la siguiente prueba necesaria.

## 1  Carga reproducible

El notebook busca los datos en tres lugares, por orden, y **solo acepta
una fuente si su huella SHA-256 coincide con la registrada en
`data/metadata.json`**:

1. archivos locales (una copia del repositorio);
2. GitHub, desde la etiqueta fijada del repositorio público;
3. el simulador integrado, que regenera exactamente los mismos datos.

Ver «no encontrado» en la primera fuente es normal en Colab. No se solicita
acceso a Google Drive. La celda del simulador aparece plegada; puede
abrirla para leer el modelo completo.

In [ ]:
#@title Simulador de referencia (no hace falta modificarlo) { display-mode: "form" }
"""Simple closed-loop actuator simulation used by the workshop.

The model is intentionally transparent. It is a teaching case, not a validated
digital twin of a particular industrial actuator.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd


SCENARIOS = {
    "nominal": {"actuator_gain": 1.0, "sensor_bias": 0.0},
    "actuator_loss": {"actuator_gain": 0.58, "sensor_bias": 0.0},
    "sensor_bias": {"actuator_gain": 1.0, "sensor_bias": 0.12},
}


@dataclass(frozen=True)
class SimulationConfig:
    dt_s: float = 0.02
    duration_s: float = 8.0
    mass_kg: float = 1.20
    damping_n_s_m: float = 2.00
    kp_n_m: float = 18.0
    kd_n_s_m: float = 5.0
    force_limit_n: float = 12.0
    position_noise_std_m: float = 0.003
    velocity_noise_std_m_s: float = 0.006
    runs_per_scenario: int = 8
    base_seed: int = 2609

    def to_dict(self) -> dict[str, float | int]:
        return asdict(self)


def reference_position(time_s: np.ndarray) -> np.ndarray:
    """Piecewise-constant reference shared by every run."""
    return np.select(
        [time_s < 1.0, time_s < 3.0, time_s < 5.0],
        [0.0, 0.50, -0.30],
        default=0.80,
    ).astype(float)


def simulate_run(
    scenario: str,
    run_index: int,
    config: SimulationConfig | None = None,
) -> pd.DataFrame:
    """Simulate one run and return logged controller and validation signals."""
    cfg = config or SimulationConfig()
    if scenario not in SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")
    if run_index < 0:
        raise ValueError("run_index must be non-negative")

    params = SCENARIOS[scenario]
    seed = cfg.base_seed + 100 * list(SCENARIOS).index(scenario) + run_index
    rng = np.random.default_rng(seed)

    time_s = np.arange(0.0, cfg.duration_s, cfg.dt_s)
    reference_m = reference_position(time_s)
    n = len(time_s)
    x_true = np.zeros(n)
    v_true = np.zeros(n)
    x_measured = np.zeros(n)
    v_measured = np.zeros(n)
    force_command = np.zeros(n)
    force_applied = np.zeros(n)

    mass = cfg.mass_kg * rng.uniform(0.96, 1.04)
    damping = cfg.damping_n_s_m * rng.uniform(0.94, 1.06)

    for k in range(n - 1):
        x_measured[k] = (
            x_true[k]
            + params["sensor_bias"]
            + rng.normal(0.0, cfg.position_noise_std_m)
        )
        v_measured[k] = v_true[k] + rng.normal(0.0, cfg.velocity_noise_std_m_s)
        raw_command = (
            cfg.kp_n_m * (reference_m[k] - x_measured[k])
            - cfg.kd_n_s_m * v_measured[k]
        )
        force_command[k] = np.clip(raw_command, -cfg.force_limit_n, cfg.force_limit_n)
        force_applied[k] = params["actuator_gain"] * force_command[k]
        acceleration = (force_applied[k] - damping * v_true[k]) / mass
        v_true[k + 1] = v_true[k] + acceleration * cfg.dt_s
        x_true[k + 1] = x_true[k] + v_true[k + 1] * cfg.dt_s

    x_measured[-1] = (
        x_true[-1]
        + params["sensor_bias"]
        + rng.normal(0.0, cfg.position_noise_std_m)
    )
    v_measured[-1] = v_true[-1] + rng.normal(0.0, cfg.velocity_noise_std_m_s)
    force_command[-1] = force_command[-2]
    force_applied[-1] = params["actuator_gain"] * force_command[-1]

    return pd.DataFrame(
        {
            "run_id": f"{scenario}_{run_index:02d}",
            "scenario": scenario,
            "seed": seed,
            "time_s": time_s,
            "reference_m": reference_m,
            "position_measured_m": x_measured,
            "velocity_measured_m_s": v_measured,
            "force_command_n": force_command,
            "position_true_m": x_true,
            "velocity_true_m_s": v_true,
            "force_applied_n": force_applied,
        }
    )


def simulate_dataset(config: SimulationConfig | None = None) -> pd.DataFrame:
    """Generate the complete balanced dataset in a stable row order."""
    cfg = config or SimulationConfig()
    if cfg.runs_per_scenario > 100:
        # Las semillas se separan 100 unidades entre escenarios.
        raise ValueError("runs_per_scenario must not exceed 100")
    frames = [
        simulate_run(scenario, run_index, cfg)
        for scenario in SCENARIOS
        for run_index in range(cfg.runs_per_scenario)
    ]
    return pd.concat(frames, ignore_index=True)

In [ ]:
import hashlib
import io
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPOSITORY = "SeRoMechatronic/2026_eu4m-ai-workshop-student"
DATA_REF = "v1.0.0"
EXPECTED_SHA256 = "faa229bd6660f8677c9c5b3e4c76cdea9b2bcc79c2cda8df61d587366a8f81a1"
PART_NAMES = [
    "actuator_signals_nominal.csv",
    "actuator_signals_actuator_loss.csv",
    "actuator_signals_sensor_bias.csv",
]


def canonical_sha256(df):
    """SHA-256 del CSV canónico: finales de línea LF y ocho decimales."""
    text = df.to_csv(index=False, float_format="%.8f", lineterminator="\n")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def from_local_files():
    # Jupyter suele ejecutar desde notebooks/; Colab, desde el directorio de carga.
    for folder in (Path("data"), Path("../data"), Path(".")):
        paths = [folder / name for name in PART_NAMES]
        if all(path.exists() for path in paths):
            return pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
    return None


def from_github():
    base = f"https://raw.githubusercontent.com/{REPOSITORY}/{DATA_REF}/data"
    frames = []
    for name in PART_NAMES:
        with urllib.request.urlopen(f"{base}/{name}", timeout=10) as response:
            frames.append(pd.read_csv(io.BytesIO(response.read())))
    return pd.concat(frames, ignore_index=True)


SOURCES = [
    ("archivos locales", from_local_files),
    (f"GitHub ({DATA_REF})", from_github),
    ("simulador integrado", simulate_dataset),
]

data = None
for source, loader in SOURCES:
    try:
        candidate = loader()
    except Exception as error:
        print(f"- {source}: no disponible ({type(error).__name__})")
        continue
    if candidate is None:
        print(f"- {source}: no encontrado")
    elif canonical_sha256(candidate) != EXPECTED_SHA256:
        print(f"- {source}: la huella SHA-256 no coincide; se descarta")
    else:
        data = candidate
        break

assert data is not None, "Ninguna fuente produjo el dataset esperado."
print(f"Fuente: {source}")
print(f"Filas: {len(data):,}")
data.head()

## 2  Contrato del dataset

Antes de interpretar una gráfica, compruebe que el archivo corresponde al
experimento descrito: 24 ejecuciones, tres escenarios equilibrados, 400
muestras por ejecución y periodo de 0,02 s. La huella SHA-256 confirma que
los valores son exactamente los del experimento de referencia.

In [ ]:
expected_columns = {
    "run_id", "scenario", "seed", "time_s", "reference_m",
    "position_measured_m", "velocity_measured_m_s", "force_command_n",
    "position_true_m", "velocity_true_m_s", "force_applied_n",
}
assert expected_columns.issubset(data.columns)
assert data["run_id"].nunique() == 24
assert data["scenario"].nunique() == 3
assert data.groupby("scenario")["run_id"].nunique().eq(8).all()
assert data.groupby("run_id").size().eq(400).all()
assert not data.isna().any().any()
assert canonical_sha256(data) == EXPECTED_SHA256
print(f"SHA-256: {canonical_sha256(data)}")
print("PASS  El archivo cumple el contrato.")

## 3  Señales representativas

Compare una ejecución de cada escenario. La posición medida representa lo
que ve el controlador. La posición real es una señal de validación disponible
únicamente porque el caso es simulado.

In [ ]:
colors = {
    "nominal": "#2A6F97",
    "actuator_loss": "#C14953",
    "sensor_bias": "#E09F3E",
}
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for scenario, color in colors.items():
    run = data[data["run_id"] == f"{scenario}_00"]
    axes[0].plot(run["time_s"], run["position_measured_m"], label=scenario, color=color)
    axes[1].plot(run["time_s"], run["position_true_m"], label=scenario, color=color)
    axes[2].plot(run["time_s"], run["force_command_n"], label=scenario, color=color)
ref = data[data["run_id"] == "nominal_00"]
axes[0].plot(ref["time_s"], ref["reference_m"], "k--", label="reference")
axes[1].plot(ref["time_s"], ref["reference_m"], "k--", label="reference")
axes[0].set_ylabel("Posición medida [m]")
axes[1].set_ylabel("Posición real [m]")
axes[2].set_ylabel("Orden de fuerza [N]")
axes[2].set_xlabel("Tiempo [s]")
axes[0].legend(ncol=4, fontsize=8)
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Pregunta breve**

En el escenario con sesgo, ¿por qué el error calculado con la posición
medida puede parecer pequeño mientras la posición real está desplazada?
Escriba su respuesta en el registro PVRD.

## 4  Características por ejecución

Se ignora el primer segundo para reducir el peso de la condición inicial.
Las características de posición real y residual sirven para validar la
simulación. Un sistema real necesitaría una medición independiente o un
modelo que produjera una referencia comparable.

In [ ]:
rows = []
for run_id, run in data.groupby("run_id"):
    g = run[run["time_s"] >= 1.0]
    measured_error = g["reference_m"] - g["position_measured_m"]
    true_error = g["reference_m"] - g["position_true_m"]
    residual = g["position_measured_m"] - g["position_true_m"]
    rows.append({
        "run_id": run_id,
        "scenario": g["scenario"].iloc[0],
        "tracking_rmse_measured_m": np.sqrt(np.mean(measured_error**2)),
        "tracking_rmse_true_m": np.sqrt(np.mean(true_error**2)),
        "mean_sensor_residual_m": residual.mean(),
        "force_command_rms_n": np.sqrt(np.mean(g["force_command_n"]**2)),
        "force_saturation_fraction": np.mean(np.abs(g["force_command_n"]) >= 11.999),
    })

features = pd.DataFrame(rows).sort_values("run_id").reset_index(drop=True)
summary = features.groupby("scenario").median(numeric_only=True)
summary.round(4)

## 5  Tres afirmaciones propuestas por una IA

1. «El escenario con mayor RMSE real mediano es el de pérdida del actuador».
2. «El sesgo puede confirmarse usando solo las señales operativas del lazo».
3. «Los resultados prueban que el método funcionará en cualquier actuador».

Para cada afirmación, indique evidencia, decisión y límite. Utilice
`aceptar`, `modificar` o `rechazar` como decisión.

In [ ]:
audit = pd.DataFrame([
    {
        "claim": 1,
        "decision": "aceptar",
        "evidence": "summary['tracking_rmse_true_m']; máximo mediano por escenario",
        "remaining_limit": (
            "Resultado del modelo sintético y de esta configuración. La diferencia con "
            "sensor_bias es menor que la dispersión entre ejecuciones; solo la "
            "diferencia con nominal es robusta"
        ),
    },
    {
        "claim": 2,
        "decision": "rechazar",
        "evidence": "El residual usa position_true_m, señal no operativa",
        "remaining_limit": "Hace falta sensor redundante, referencia externa o modelo validado",
    },
    {
        "claim": 3,
        "decision": "rechazar",
        "evidence": "Solo 24 ejecuciones simuladas de un modelo didáctico",
        "remaining_limit": "No hay validación con otros actuadores, cargas o datos reales",
    },
])
audit

## 6  Pruebas de las decisiones

La primera prueba confirma una propiedad limitada al dataset y muestra hasta
dónde llega: la pérdida del actuador supera siempre al nominal, pero sus
rangos se solapan con los del sesgo del sensor. La segunda muestra que la
señal necesaria para confirmar el sesgo pertenece a la validación de la
simulación.

In [ ]:
largest = summary["tracking_rmse_true_m"].idxmax()
assert largest == "actuator_loss"

by_scenario = {
    s: features.loc[features["scenario"] == s, "tracking_rmse_true_m"]
    for s in ("nominal", "actuator_loss", "sensor_bias")
}
# Separación robusta frente al nominal...
assert by_scenario["actuator_loss"].min() > by_scenario["nominal"].max()
# ...pero sin separación frente a sensor_bias: los rangos se solapan.
assert by_scenario["actuator_loss"].min() < by_scenario["sensor_bias"].max()

operational_columns = {
    "time_s", "reference_m", "position_measured_m",
    "velocity_measured_m_s", "force_command_n",
}
validation_only = {"position_true_m", "velocity_true_m_s", "force_applied_n"}
assert operational_columns.issubset(data.columns)
assert validation_only.issubset(data.columns)
print("PASS  Las pruebas respaldan las decisiones documentadas.")

## 7  Interpretación orientativa

La pérdida del actuador aumenta el error real en esta configuración, sin
ambigüedad frente al escenario nominal. Frente al sesgo del sensor la
diferencia es demasiado pequeña para sostener un orden fiable. El sesgo
resulta visible al comparar medida y posición real, pero esa comparación
no estaría disponible con un único sensor real. La conclusión responsable
propone una medición independiente y evita generalizar desde un caso
sintético.